# Notebook 06: Comprehensive Model Comparison (All 3 Principal Components)

This notebook evaluates the same LOSO benchmark across two target families: a toy mean-ratings target for quick experimentation, and the proper Daly PCA targets representing Valence, Energy, and Tension. For each target family, the models are compared on three feature subsets: both EEG + audio, EEG only, and audio only.

In [1]:
import os
import glob
import logging
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.base import clone
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
)
logger = logging.getLogger('emotion_benchmark')

/home/trainerblue/Documents/daly-comp-analysis/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load features
base_path = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.split(os.getcwd())[-1] == 'notebooks' else os.getcwd()
features_dir = os.path.join(base_path, 'data', 'features')
csv_files = glob.glob(os.path.join(features_dir, 'sub-*_features.csv'))
full_df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print(f"Total trials collected: {len(full_df)}")

# Build the two target families
rating_cols = [f'Q{q}' for q in range(800, 808)]
imputer = KNNImputer(n_neighbors=5, weights='distance')
imputed_ratings = imputer.fit_transform(full_df[rating_cols])

full_df['Mean_Rating_Toy'] = imputed_ratings.mean(axis=1)

pca = PCA(n_components=3)
target_b_pca = pca.fit_transform(imputed_ratings)
full_df['PC1_Valence_Subj'] = target_b_pca[:, 0]
full_df['PC2_Energy_Subj'] = target_b_pca[:, 1]
full_df['PC3_Tension_Subj'] = target_b_pca[:, 2]
print(f"Explained Variance by 3 PCs: {sum(pca.explained_variance_ratio_):.2f}")

# Separate EEG and audio feature groups
meta_cols = {
    'track_id', 'run', 'subject', 'number', 'valence', 'energy', 'tension',
    'anger', 'fear', 'happy', 'sad', 'tender', 'TARGET',
    'Mean_Rating_Toy', 'PC1_Valence_Subj', 'PC2_Energy_Subj', 'PC3_Tension_Subj',
} | set(rating_cols)

eeg_prefixes = (
    'FP1_', 'FP2_', 'F7_', 'F3_', 'Fz_', 'F4_', 'F8_',
    'T3_', 'C3_', 'Cz_', 'C4_', 'T4_', 'T5_', 'P3_', 'Pz_', 'P4_', 'T6_', 'O1_', 'O2_'
)
audio_prefixes = ('mfcc_', 'chroma_', 'spectral_', 'rms_energy')

feature_columns = [c for c in full_df.columns if c not in meta_cols]
eeg_columns = [c for c in feature_columns if c.startswith(eeg_prefixes)]
audio_columns = [c for c in feature_columns if c.startswith(audio_prefixes)]
both_columns = sorted(set(eeg_columns) | set(audio_columns))

feature_sets = {
    'Both': both_columns,
    'EEG only': eeg_columns,
    'Audio only': audio_columns,
}

print(f"Feature counts -> Both: {len(both_columns)}, EEG only: {len(eeg_columns)}, Audio only: {len(audio_columns)}")

subjects = full_df['subject'].unique()
models = {
    'Lasso': Lasso(alpha=0.1, random_state=42, max_iter=10000),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
    ),
}

def evaluate_loso(estimator, X_frame, y_series, model_name, target_name, feature_name):
    actuals = []
    predictions = []

    fold_bar = tqdm(subjects, desc=f"{target_name} | {model_name} | {feature_name}", leave=False)
    for test_sub in fold_bar:
        train_idx = full_df['subject'] != test_sub
        test_idx = full_df['subject'] == test_sub

        X_train = X_frame.loc[train_idx]
        X_test = X_frame.loc[test_idx]
        y_train = y_series.loc[train_idx]
        y_test = y_series.loc[test_idx]

        fitted_model = clone(estimator)

        if model_name == 'Lasso':
            scaler = StandardScaler()
            X_train_fit = scaler.fit_transform(X_train)
            X_test_fit = scaler.transform(X_test)
        else:
            X_train_fit = X_train.values
            X_test_fit = X_test.values

        fitted_model.fit(X_train_fit, y_train)
        fold_predictions = fitted_model.predict(X_test_fit)

        actuals.extend(y_test.to_numpy())
        predictions.extend(np.asarray(fold_predictions))

    actuals = np.asarray(actuals)
    predictions = np.asarray(predictions)

    rmse = float(np.sqrt(mean_squared_error(actuals, predictions)))
    r2 = float(r2_score(actuals, predictions))
    pearson_r = float(np.corrcoef(actuals, predictions)[0, 1])

    return {
        'RMSE': rmse,
        'R2': r2,
        'Pearson r': pearson_r,
    }

def build_results_table(target_map, family_name):
    rows = []

    logger.info('Starting benchmark family: %s', family_name)
    for target_name, y_series in target_map.items():
        logger.info('Target: %s', target_name)
        for model_name, estimator in models.items():
            logger.info('Model: %s', model_name)
            row = {
                'Family': family_name,
                'Target': target_name,
                'Model': model_name,
            }

            for feature_name, columns in feature_sets.items():
                logger.info('Evaluating %s / %s / %s', target_name, model_name, feature_name)
                metrics = evaluate_loso(
                    estimator=estimator,
                    X_frame=full_df[columns],
                    y_series=y_series,
                    model_name=model_name,
                    target_name=target_name,
                    feature_name=feature_name,
                )
                row[f'{feature_name} RMSE'] = metrics['RMSE']
                row[f'{feature_name} R2'] = metrics['R2']
                row[f'{feature_name} r'] = metrics['Pearson r']

            rows.append(row)

    results = pd.DataFrame(rows)
    summary = results[[
        'Family', 'Target', 'Model',
        'Both r', 'EEG only r', 'Audio only r',
    ]].copy()
    return results, summary

mean_target_map = {
    'Mean_Rating_Toy': full_df['Mean_Rating_Toy'],
}

daly_target_map = {
    'PC1_Valence_Subj': full_df['PC1_Valence_Subj'],
    'PC2_Energy_Subj': full_df['PC2_Energy_Subj'],
    'PC3_Tension_Subj': full_df['PC3_Tension_Subj'],
}

mean_results, mean_summary = build_results_table(mean_target_map, 'Mean ratings experiment')
daly_results, daly_summary = build_results_table(daly_target_map, 'Daly PCA targets')

print('\n=== Mean ratings experiment: Pearson summary ===')
display(mean_summary)
print('\n=== Mean ratings experiment: full metrics ===')
display(mean_results)

print('\n=== Daly PCA targets: Pearson summary ===')
display(daly_summary)
print('\n=== Daly PCA targets: full metrics ===')
display(daly_results)

2026-05-23 11:42:02,381 INFO Starting benchmark family: Mean ratings experiment
2026-05-23 11:42:02,382 INFO Target: Mean_Rating_Toy
2026-05-23 11:42:02,383 INFO Model: Lasso
2026-05-23 11:42:02,384 INFO Evaluating Mean_Rating_Toy / Lasso / Both


Total trials collected: 990
Explained Variance by 3 PCs: 0.44
Feature counts -> Both: 199, EEG only: 133, Audio only: 66


2026-05-23 11:42:03,536 INFO Evaluating Mean_Rating_Toy / Lasso / EEG only     
2026-05-23 11:42:04,391 INFO Evaluating Mean_Rating_Toy / Lasso / Audio only       
2026-05-23 11:42:04,975 INFO Model: Random Forest                                    
2026-05-23 11:42:04,977 INFO Evaluating Mean_Rating_Toy / Random Forest / Both
2026-05-23 11:42:24,869 INFO Evaluating Mean_Rating_Toy / Random Forest / EEG only     
2026-05-23 11:42:34,042 INFO Evaluating Mean_Rating_Toy / Random Forest / Audio only       
2026-05-23 11:42:47,740 INFO Model: XGBoost                                                  
2026-05-23 11:42:47,741 INFO Evaluating Mean_Rating_Toy / XGBoost / Both
2026-05-23 11:43:45,834 INFO Evaluating Mean_Rating_Toy / XGBoost / EEG only     
2026-05-23 11:44:19,619 INFO Evaluating Mean_Rating_Toy / XGBoost / Audio only       
2026-05-23 11:44:30,745 INFO Starting benchmark family: Daly PCA targets               
2026-05-23 11:44:30,746 INFO Target: PC1_Valence_Subj
2026-05-23 11:


=== Mean ratings experiment: Pearson summary ===


,Family,Target,Model,Both r,EEG only r,Audio only r
0,Mean ratings experiment,Mean_Rating_Toy,Lasso,-0.429675,-0.429675,-0.436001
1,Mean ratings experiment,Mean_Rating_Toy,Random Forest,-0.031243,-0.114883,0.125468
2,Mean ratings experiment,Mean_Rating_Toy,XGBoost,-0.028612,-0.106522,0.104234


...

,Family,Target,Model,Both RMSE,Both R2,Both r,EEG only RMSE,EEG only R2,EEG only r,Audio only RMSE,Audio only R2,Audio only r
0,Mean ratings experiment,Mean_Rating_Toy,Lasso,0.748116,-0.016953,-0.429675,0.748116,-0.016953,-0.429675,0.747485,-0.015236,-0.436001
1,Mean ratings experiment,Mean_Rating_Toy,Random Forest,0.774573,-0.090153,-0.031243,0.787880,-0.127932,-0.114883,0.774596,-0.090219,0.125468
2,Mean ratings experiment,Mean_Rating_Toy,XGBoost,0.806557,-0.182041,-0.028612,0.824440,-0.235039,-0.106522,0.798743,-0.159249,0.104234



=== Daly PCA targets: Pearson summary ===


,Family,Target,Model,Both r,EEG only r,Audio only r
0,Daly PCA targets,PC1_Valence_Subj,Lasso,0.157448,0.043552,0.163446
1,Daly PCA targets,PC1_Valence_Subj,Random Forest,0.336502,0.035796,0.374955
2,Daly PCA targets,PC1_Valence_Subj,XGBoost,0.300045,0.057484,0.357386
3,Daly PCA targets,PC2_Energy_Subj,Lasso,0.085996,0.016494,0.079735
4,Daly PCA targets,PC2_Energy_Subj,Random Forest,0.225616,-0.054548,0.216473
5,Daly PCA targets,PC2_Energy_Subj,XGBoost,0.127669,-0.023224,0.176426
6,Daly PCA targets,PC3_Tension_Subj,Lasso,0.210816,0.017395,0.200811
7,Daly PCA targets,PC3_Tension_Subj,Random Forest,0.281387,0.015609,0.262349
8,Daly PCA targets,PC3_Tension_Subj,XGBoost,0.177660,-0.036277,0.220695



=== Daly PCA targets: full metrics ===


,Family,Target,Model,Both RMSE,Both R2,Both r,EEG only RMSE,EEG only R2,EEG only r,Audio only RMSE,Audio only R2,Audio only r
0,Daly PCA targets,PC1_Valence_Subj,Lasso,2.410843,0.024456,0.157448,2.445506,-0.003798,0.043552,2.409153,0.025824,0.163446
1,Daly PCA targets,PC1_Valence_Subj,Random Forest,2.308865,0.105242,0.336502,2.545407,-0.087485,0.035796,2.323936,0.093523,0.374955
2,Daly PCA targets,PC1_Valence_Subj,XGBoost,2.368646,0.058308,0.300045,2.560904,-0.100767,0.057484,2.390993,0.040455,0.357386
3,Daly PCA targets,PC2_Energy_Subj,Lasso,2.245384,0.005959,0.085996,2.254482,-0.002113,0.016494,2.245614,0.005755,0.079735
4,Daly PCA targets,PC2_Energy_Subj,Random Forest,2.217158,0.030794,0.225616,2.348556,-0.087489,-0.054548,2.310458,-0.052493,0.216473
5,Daly PCA targets,PC2_Energy_Subj,XGBoost,2.354865,-0.093339,0.127669,2.417714,-0.152478,-0.023224,2.415113,-0.150000,0.176426
6,Daly PCA targets,PC3_Tension_Subj,Lasso,2.160120,0.041089,0.210816,2.206650,-0.000668,0.017395,2.164637,0.037074,0.200811
7,Daly PCA targets,PC3_Tension_Subj,Random Forest,2.131269,0.066532,0.281387,2.250839,-0.041146,0.015609,2.218858,-0.011770,0.262349
8,Daly PCA targets,PC3_Tension_Subj,XGBoost,2.244550,-0.035337,0.177660,2.340362,-0.125613,-0.036277,2.305194,-0.092039,0.220695
